<a href="https://colab.research.google.com/github/binary-cpu/port-tracker/blob/main/02_build_map_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Port Traffic: Map Data Preparation

**Data snapshot**: through 2026-10-09, accessed 2026-10-18. Source: UN Global Platform; IMF PortWatch.

## A. Build the Port and Country Data Files

In [1]:
import requests, json, time, zipfile, os, pandas as pd
from concurrent.futures import ThreadPoolExecutor
DAILY = "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/Daily_Ports_Data/FeatureServer/0/query"
PORTS = "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/PortWatch_ports_database/FeatureServer/0/query"
TYPES = ["container", "dry_bulk", "general_cargo", "roro", "tanker"]

def query(url, params):
    r = requests.get(url, params={"returnGeometry": "false", "f": "json", **params}, timeout=180).json()
    if "error" in r:
        raise RuntimeError(r["error"])
    return r

def fetch_pages(url, params, page=1000):
    rows, off = [], 0
    while True:
        feats = query(url, {**params, "resultOffset": off, "resultRecordCount": page}).get("features", [])
        rows += [f["attributes"] for f in feats]
        if len(feats) < page:
            break
        off += page
    return pd.DataFrame(rows)

def to_date(v):
    return pd.to_datetime(v, unit="ms") if isinstance(v, (int, float)) else pd.to_datetime(v)

def sums(cols):
    return json.dumps([{"statisticType": "sum", "onStatisticField": c, "outStatisticFieldName": f"sum_{c}"} for c in cols])

def port_totals(start, end):
    cols = ["portcalls"] + [f"portcalls_{t}" for t in TYPES]
    d = fetch_pages(DAILY, {"where": f"date >= '{start}' AND date <= '{end}'", "groupByFieldsForStatistics": "portid", "outStatistics": sums(cols), "orderByFields": "portid"})
    return d.rename(columns={f"sum_{c}": c for c in cols})

f = query(DAILY, {"where": "1=1", "outFields": "date", "orderByFields": "date DESC", "resultRecordCount": 1})["features"]
LAST = to_date(f[0]["attributes"]["date"]).normalize()
print("last day:", LAST.date())

# ---- B. port coordinates and the join check ----
pdb = fetch_pages(PORTS, {"where": "1=1", "outFields": "portid,portname,country,ISO3,continent,lat,lon", "orderByFields": "ObjectId"})
print("ports in the database:", len(pdb), "| unique ids:", pdb.portid.nunique(), "| missing coordinates:", int(pdb[["lat", "lon"]].isna().any(axis=1).sum()),
      "| out of range:", int(((pdb.lat.abs() > 90) | (pdb.lon.abs() > 180)).sum()))

# ---- C. summary per port: latest 28 and 90 days against the same days 364 days earlier ----
def window(days):
    r_end, r_start = LAST, LAST - pd.Timedelta(days=days - 1)
    return (r_start, r_end), (r_start - pd.Timedelta(days=364), r_end - pd.Timedelta(days=364))
(r28, b28), (r90, b90) = window(28), window(90)
rec28, bas28 = port_totals(r28[0].date(), r28[1].date()), port_totals(b28[0].date(), b28[1].date())
rec90, bas90 = port_totals(r90[0].date(), r90[1].date()), port_totals(b90[0].date(), b90[1].date())
summary = pdb.copy()
for tag, rec, bas in [("28", rec28, bas28), ("90", rec90, bas90)]:
    summary = summary.merge(bas[["portid", "portcalls"]].rename(columns={"portcalls": f"base_{tag}"}), on="portid", how="left")
    summary = summary.merge(rec[["portid", "portcalls"]].rename(columns={"portcalls": f"recent_{tag}"}), on="portid", how="left")
for t in TYPES:
    summary = summary.merge(bas28[["portid", f"portcalls_{t}"]].rename(columns={f"portcalls_{t}": f"base_28_{t}"}), on="portid", how="left")
    summary = summary.merge(rec28[["portid", f"portcalls_{t}"]].rename(columns={f"portcalls_{t}": f"recent_28_{t}"}), on="portid", how="left")
print("ports with no daily totals:", int(summary.recent_28.isna().sum()), "| daily ports missing from the database:", len(set(rec28.portid) - set(pdb.portid)))

# ---- D. calls per day by country and month, last 12 complete months ----
months = pd.period_range(LAST.to_period("M") - 12, LAST.to_period("M") - 1, freq="M")
cache = {}
def month_totals(p):
    t0 = time.time()
    t = fetch_pages(DAILY, {"where": f"year = {p.year} AND month = {p.month}", "groupByFieldsForStatistics": "ISO3", "outStatistics": sums(["portcalls"])})
    t["month_start"] = p.to_timestamp(); t["days"] = p.days_in_month; cache[p] = t
    print(f"{p} done in {time.time() - t0:.0f}s", flush=True)
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(month_totals, months))
cm = pd.concat([cache[p] for p in months]).rename(columns={"sum_portcalls": "calls"})
cm["calls_per_day"] = (cm.calls / cm.days).round(2)
cm = cm.merge(pdb.drop_duplicates("ISO3")[["ISO3", "country"]], on="ISO3", how="left")[["month_start", "ISO3", "country", "calls_per_day"]]

# ---- E. write the data files and a zip to download ----
os.makedirs("data", exist_ok=True)
summary.to_csv("data/port_summary.csv", index=False); cm.to_csv("data/country_monthly.csv", index=False)
info = {"last_day": str(LAST.date()), "recent_28": [str(r28[0].date()), str(r28[1].date())], "baseline_28": [str(b28[0].date()), str(b28[1].date())],
        "recent_90": [str(r90[0].date()), str(r90[1].date())], "baseline_90": [str(b90[0].date()), str(b90[1].date())],
        "source": "UN Global Platform; IMF PortWatch (AIS-observed port calls). Derived summaries only."}
json.dump(info, open("data/data_info.json", "w"), indent=2)
with zipfile.ZipFile("app_data.zip", "w") as z:
    for fn in ["port_summary.csv", "country_monthly.csv", "data_info.json"]:
        z.write(f"data/{fn}")
print("\nrows:", len(summary), "| columns:", summary.shape[1], "| country-month rows:", len(cm), "| zip KB:", os.path.getsize("app_data.zip") // 1024)
try:
    from google.colab import files; files.download("app_data.zip")
except Exception:
    print("Not in Colab: app_data.zip is next to the notebook.")

last day: 2026-10-09
ports in the database: 2065 | unique ids: 2065 | missing coordinates: 0 | out of range: 0
ports with no daily totals: 0 | daily ports missing from the database: 0
2025-12 done in 0s
2025-10 done in 0s
2026-01 done in 0s
2025-11 done in 0s
2026-04 done in 0s
2026-05 done in 0s
2026-02 done in 1s
2026-03 done in 1s
2026-07 done in 0s
2026-09 done in 0s
2026-08 done in 0s
2026-06 done in 1s

rows: 2065 | columns: 21 | country-month rows: 2160 | zip KB: 284


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**What the files contain.** `port_summary.csv` has one row per port (2,065): identifiers, country, continent, latitude and longitude, and port calls in the latest 28 and 90 days (`recent_28`, `recent_90`) and in the same days 364 days earlier (`base_28`, `base_90`), plus the 28-day calls by ship type. `country_monthly.csv` has calls per day by country for the last 12 complete months. `data_info.json` records the data date and window dates. Checks: 2,065 ports with unique ids, no missing or out-of-range coordinates, every port in the daily data found in the ports database, and no ports without daily totals. All counts are AIS-observed port calls; files hold derived summaries only.